# Phase 06B.03 — Traffic temporal grounding preflight

Locks schema-v2 encoder, support and traffic-extractor provenance. It does not train or evaluate validation.

In [ ]:
from pathlib import Path
import json,sys
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"src").is_dir()),None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from roadbuddy_common import save_json
from phase06a_common import sha256_file,sha256_json
from phase06b_common import assert_selected_track,validate_temporal_input_manifest
P=ROOT/"outputs/phase06b/novelty_protocol/novelty_protocol.json"; OUT=ROOT/"outputs/phase06b/traffic_temporal_grounding/preflight"; OUT.mkdir(parents=True,exist_ok=True)
IM=ROOT/"outputs/phase06b/temporal_inputs/temporal_input_manifest.json"
if not P.is_file(): raise RuntimeError("Lock Phase06B_01 first")
protocol=json.loads(P.read_text())
if protocol.get("selected_track")!="traffic_temporal_grounding":
    save_json(OUT/"PHASE06B_03_STATUS.json",{"status":"not_selected","selected_track":protocol.get("selected_track")})
    raise RuntimeError("Temporal branch was not selected")
assert_selected_track(protocol,"traffic_temporal_grounding")


In [ ]:
if not IM.is_file():
    template={"schema_version":2,"visual_encoder":"","visual_encoder_revision":"","visual_encoder_checkpoint_sha256":"","question_encoder":"","question_encoder_revision":"","question_encoder_checkpoint_sha256":"","preprocessing_sha256":"","dtype":"","normalization":"","candidate_count":32,"support_annotation_split":"train","support_annotations_path":"data/phase06b/train_temporal_support.csv","support_annotations_sha256":"","feature_bank_schema_version":2,"split_membership_sha256":"","traffic_extractor":{"feature_vocabulary":[],"model_id":"","model_revision":"","checkpoint_sha256":"","confidence_thresholds":{},"aggregation_policy":"","output_dimension":0,"missing_detection_behavior":"","license":"","provenance":""}}
    save_json(ROOT/"outputs/phase06b/temporal_inputs/temporal_input_manifest.template.json",template)
    save_json(OUT/"PHASE06B_03_STATUS.json",{"status":"awaiting_temporal_input_manifest","schema_version":2})
    raise RuntimeError("Freeze encoder/support/traffic provenance")
inputs=validate_temporal_input_manifest(json.loads(IM.read_text())); support=ROOT/inputs["support_annotations_path"]
if not support.is_file() or sha256_file(support)!=inputs["support_annotations_sha256"]: raise ValueError("Support file/hash mismatch")
locked={"schema_version":2,"status":"locked","selected_track":"traffic_temporal_grounding","parent_protocol_sha256":sha256_json(protocol),"input_manifest_sha256":sha256_file(IM),"candidate_count":32,"total_visual_tile_budget":8,"checkpoint_selection":"group-safe train_fit/inner_dev only","stage_b":"independent reset and full-train retrain to locked optimizer step","validation_support_as_selector_input":False,"oracle_role":"diagnostic_only"}
save_json(OUT/"traffic_temporal_grounding_protocol.json",locked); save_json(OUT/"PHASE06B_03_STATUS.json",{"status":"complete","protocol_sha256":sha256_json(locked)})
locked
